In [1]:
"""
Multi-Classifier Pipeline: XGBoost, CatBoost, LightGBM
-------------------------------------------------------
Predicts `ClassID` from Longitude, Latitude, B04, B03, B08, B02, B01, B12, B11, EVI, NDBI, MNDWI, BSI, NDVI.
Dataset: data.csv
"""

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import NearestCentroid
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
import warnings
import gc
import os
import joblib

warnings.filterwarnings("ignore")

# ──────────────────────────────────────────────
# 1. DATA LOADING & PREPROCESSING
# ──────────────────────────────────────────────

def load_data(path: str) -> pd.DataFrame:
    """Load CSV or Parquet and strip whitespace from column names."""
    if path.endswith('.parquet') or path.endswith('.parq'):
        df = pd.read_parquet(path)
    else:
        df = pd.read_csv(path)
    df.columns = df.columns.str.strip()
    return df


def load_multiple_parquets(file_paths: list[str], rows_per_file: int = None, sample_frac: float = None, target_col: str = "ClassID", stratified: bool = True) -> pd.DataFrame:
    """
    Load multiple parquet files, optionally sampling from each, and concatenate.
    
    Parameters
    ----------
    file_paths : list[str]
        List of paths to parquet files
    rows_per_file : int, optional
        Number of rows to take from each file (if None, uses sample_frac)
    sample_frac : float, optional
        Fraction of rows to sample from each file (0.0-1.0). Default: 0.1 (10%)
    target_col : str
        Column name for target variable (used for stratified sampling)
    stratified : bool
        If True and rows_per_file is set, samples equally from each class
    
    Returns
    -------
    pd.DataFrame
        Concatenated dataframe from all files
    """
    if sample_frac is None and rows_per_file is None:
        sample_frac = 0.1  # Default: 10% of each file
    
    dfs = []
    for path in file_paths:
        print(f"Loading {path}...")
        df = load_data(path)
        
        if rows_per_file is not None:
            if stratified and target_col in df.columns:
                # Sample equally from each class (3 classes)
                rows_per_class = rows_per_file // 3
                sampled_dfs = []
                
                for class_id in df[target_col].unique():
                    class_df = df[df[target_col] == class_id]
                    # Take min of (rows_per_class, available rows in this class)
                    n_samples = min(rows_per_class, len(class_df))
                    sampled = class_df.sample(n=n_samples, random_state=42)
                    sampled_dfs.append(sampled)
                    print(f"    Class {class_id}: sampled {len(sampled)} rows")
                
                df = pd.concat(sampled_dfs, ignore_index=True)
            else:
                df = df.head(rows_per_file)
        elif sample_frac is not None:
            df = df.sample(frac=sample_frac, random_state=42)
        
        dfs.append(df)
        print(f"  → Total from file: {len(df)} rows\n")
    
    combined_df = pd.concat(dfs, ignore_index=True)
    print(f"Total combined data: {len(combined_df)} rows")
    print(f"Class distribution:\n{combined_df[target_col].value_counts()}\n")
    return combined_df


def add_index(df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute spectral indices and append them as new columns.

    Requires columns: B02, B03, B04, B08, B11.

    Added columns
    --------------
    EVI   = 2.5 * (B08 - B04) / (B08 + 6*B04 - 7.5*B02 + 1)
    NDBI  = (B11 - B08) / (B11 + B08)
    MNDWI = (B03 - B11) / (B03 + B11)
    BSI   = (B03 + B08) / (B03 - B08)
    NDVI  = (B08 - B04) / (B08 + B04)
    """
    df = df.copy()

    B02 = df["B02"]
    B03 = df["B03"]
    B04 = df["B04"]
    B08 = df["B08"]
    B11 = df["B11"]

    # EVI – Enhanced Vegetation Index
    evi_denom = B08 + 6.0 * B04 - 7.5 * B02 + 1.0
    df["EVI"] = np.where(
        evi_denom != 0,
        2.5 * (B08 - B04) / evi_denom,
        0.0,
    )

    # NDBI – Normalized Difference Built-up Index
    ndbi_denom = B11 + B08
    df["NDBI"] = np.where(
        ndbi_denom != 0,
        (B11 - B08) / ndbi_denom,
        0.0,
    )

    # MNDWI – Modified Normalized Difference Water Index
    mndwi_denom = B03 + B11
    df["MNDWI"] = np.where(
        mndwi_denom != 0,
        (B03 - B11) / mndwi_denom,
        0.0,
    )

    # BSI – Bare Soil Index  (Green + NIR) / (Green - NIR)
    bsi_denom = B03 - B08
    df["BSI"] = np.where(
        bsi_denom != 0,
        (B03 + B08) / bsi_denom,
        0.0,
    )

    # NDVI – Normalized Difference Vegetation Index  (NIR - RED) / (NIR + RED)
    ndvi_denom = B08 + B04
    df["NDVI"] = np.where(
        ndvi_denom != 0,
        (B08 - B04) / ndvi_denom,
        0.0,
    )

    return df


def preprocess(
    df: pd.DataFrame,
    feature_cols: list[str],
    target_col: str,
    test_size: float = 0.2,
    random_state: int = 42,
    scale: bool = True,
):
    """
    Encode target, optionally scale features, and split into train/test.

    Returns
    -------
    X_train, X_test, y_train, y_test, label_encoder, scaler
    """
    X = df[feature_cols].copy()
    y = df[target_col].copy()

    # Encode target labels → 0-based integers
    le = LabelEncoder()
    y = le.fit_transform(y)

    # Optional feature scaling
    scaler = None
    if scale:
        scaler = StandardScaler()
        X = pd.DataFrame(scaler.fit_transform(X), columns=feature_cols)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )
    return X_train, X_test, y_train, y_test, le, scaler


# ──────────────────────────────────────────────
# 2. MODEL DEFINITIONS
# ──────────────────────────────────────────────

def build_models(num_classes: int, random_state: int = 42) -> dict:
    """Return a dict of name → classifier instance."""
    objective = "multi:softmax" if num_classes > 2 else "binary:logistic"

    models = {
        "XGBoost": XGBClassifier(
            n_estimators=200,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            objective=objective,
            num_class=num_classes if num_classes > 2 else None,
            eval_metric="mlogloss" if num_classes > 2 else "logloss",
            use_label_encoder=False,
            random_state=random_state,
            tree_method="hist",
            verbosity=0,
        ),
        "CatBoost": CatBoostClassifier(
            iterations=200,
            depth=6,
            learning_rate=0.05,
            loss_function="MultiClass" if num_classes > 2 else "Logloss",
            random_seed=random_state,
            verbose=0,
        ),
        "LightGBM": LGBMClassifier(
            n_estimators=200,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            num_class=num_classes if num_classes > 2 else None,
            objective="multiclass" if num_classes > 2 else "binary",
            random_state=random_state,
            verbose=-1,
        ),
        "CART": DecisionTreeClassifier(
            max_depth=10,
            min_samples_split=5,
            min_samples_leaf=2,
            random_state=random_state,
        ),
        # "RandomForest": RandomForestClassifier(
        #     n_estimators=200,
        #     max_depth=10,
        #     min_samples_split=5,
        #     min_samples_leaf=2,
        #     random_state=random_state,
        #     n_jobs=1,
        # ),
        # "MinDistance": NearestCentroid(),
    }
    return models


# ──────────────────────────────────────────────
# 3. TRAINING & EVALUATION
# ──────────────────────────────────────────────

def train_model(model, X_train, y_train):
    """Fit a classifier and return it."""
    model.fit(X_train, y_train)
    return model


def evaluate_model(model, X_test, y_test, label_encoder, model_name: str):
    """Print accuracy, classification report, and confusion matrix."""
    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    target_names = [str(c) for c in label_encoder.classes_]

    print(f"\n{'='*55}")
    print(f"  {model_name}")
    print(f"{'='*55}")
    print(f"  Accuracy : {acc:.4f}")
    print(f"\n  Classification Report:\n")
    print(classification_report(y_test, y_pred, target_names=target_names))
    print(f"  Confusion Matrix:\n{confusion_matrix(y_test, y_pred)}\n")

    return {"name": model_name, "accuracy": acc, "predictions": y_pred}


def cross_validate_model(model, X, y, cv: int = 5):
    """Return mean and std of cross-validation accuracy."""
    scores = cross_val_score(model, X, y, cv=cv, scoring="accuracy")
    return scores.mean(), scores.std()


def save_model(model, model_name: str, save_dir: str = "saved_models"):
    """Save a trained model to disk using joblib."""
    os.makedirs(save_dir, exist_ok=True)
    filename = f"{model_name.replace(' ', '_').lower()}.joblib"
    filepath = os.path.join(save_dir, filename)
    joblib.dump(model, filepath)
    print(f"  Model saved → {filepath}")
    return filepath


# ──────────────────────────────────────────────
# 4. COMPARISON SUMMARY
# ──────────────────────────────────────────────

def compare_results(results: list[dict]):
    """Print a side-by-side accuracy comparison."""
    print(f"\n{'='*55}")
    print("  MODEL COMPARISON")
    print(f"{'='*55}")
    for r in sorted(results, key=lambda x: x["accuracy"], reverse=True):
        print(f"  {r['name']:<12}  Accuracy: {r['accuracy']:.4f}")
    print()


# ──────────────────────────────────────────────
# 5. MAIN PIPELINE
# ──────────────────────────────────────────────

def main():
    # --- Configuration ---
    PARQUET_DIR = "/kaggle/input/datasets/mezbaussalaheen/filtered-sentinel2-dataset-bangladesh/"
    PARQUET_FILES = [f"{PARQUET_DIR}data{i}.parquet" for i in range(1, 13)]  # data1.parquet to data12.parquet
    
    # Use SAMPLE_FRAC instead of ROWS_PER_FILE to avoid memory issues
    SAMPLE_FRAC = 0.05  # Sample 5% from each file (adjust based on available memory)
    # ROWS_PER_FILE = 200  # Alternative: use fixed rows per file if preferred
    
    FEATURE_COLS = ["B04", "B03", "B08", "B02", "B01", "B12", "B11", "EVI", "NDBI", "MNDWI", "BSI", "NDVI"]
    TARGET_COL = "ClassID"
    TEST_SIZE = 0.2
    RANDOM_STATE = 42
    SCALE = True          # set False to skip StandardScaler
    CV_FOLDS = 3          # cross-validation folds

    # --- Load & preprocess ---
    print("Loading and combining parquet files...\n")
    df = load_multiple_parquets(PARQUET_FILES, sample_frac=SAMPLE_FRAC, target_col=TARGET_COL, stratified=True)
    df = add_index(df)
    print(f"Dataset shape : {df.shape}")
    print(f"Class distribution:\n{df[TARGET_COL].value_counts()}\n")

    X_train, X_test, y_train, y_test, le, scaler = preprocess(
        df, FEATURE_COLS, TARGET_COL,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        scale=SCALE,
    )
    del df  # free raw data
    gc.collect()

    num_classes = len(le.classes_)
    models = build_models(num_classes, RANDOM_STATE)

    # --- Train, evaluate, cross-validate (one at a time to save memory) ---
    results = []
    for name, model in models.items():
        print(f"\n>>> Training {name} ...")
        model = train_model(model, X_train, y_train)
        res = evaluate_model(model, X_test, y_test, le, name)
        results.append(res)

        cv_mean, cv_std = cross_validate_model(model, X_train, y_train, cv=CV_FOLDS)
        print(f"  {CV_FOLDS}-Fold CV  : {cv_mean:.4f} ± {cv_std:.4f}")

        # Save trained model to disk
        save_model(model, name)

        # Free model memory before loading next one
        del model
        gc.collect()

    # Save label encoder and scaler for inference
    save_model(le, "label_encoder")
    if scaler is not None:
        save_model(scaler, "scaler")

    # --- Summary ---
    compare_results(results)


if __name__ == "__main__":
    main()


Loading and combining parquet files...

Loading /kaggle/input/datasets/mezbaussalaheen/filtered-sentinel2-dataset-bangladesh/data1.parquet...
  → Total from file: 5958142 rows

Loading /kaggle/input/datasets/mezbaussalaheen/filtered-sentinel2-dataset-bangladesh/data2.parquet...
  → Total from file: 5950640 rows

Loading /kaggle/input/datasets/mezbaussalaheen/filtered-sentinel2-dataset-bangladesh/data3.parquet...
  → Total from file: 3252676 rows

Loading /kaggle/input/datasets/mezbaussalaheen/filtered-sentinel2-dataset-bangladesh/data4.parquet...
  → Total from file: 4555071 rows

Loading /kaggle/input/datasets/mezbaussalaheen/filtered-sentinel2-dataset-bangladesh/data5.parquet...
  → Total from file: 5939384 rows

Loading /kaggle/input/datasets/mezbaussalaheen/filtered-sentinel2-dataset-bangladesh/data6.parquet...
  → Total from file: 6011331 rows

Loading /kaggle/input/datasets/mezbaussalaheen/filtered-sentinel2-dataset-bangladesh/data7.parquet...
  → Total from file: 5632918 rows

L